# MyoChallenge 2026 — ChaseTag quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MyoHub/myosuite/blob/fix/tutorial-5.4-mm-10m-2-models-version/tutorials/5.6_ChaseTag_MyoChallenge.ipynb)

For **first-time invitees**. About 10 minutes:

1. Register on the **website** (not in this notebook)
2. Copy your **CLI token** from Submissions
3. Export the [tutorial 5.4](https://colab.research.google.com/github/MyoHub/myosuite/blob/dev/tutorials/5.4_MuscleMimic_Directional_Locomotion.ipynb) walker to ONNX and submit it
4. Watch the result on the **website**
5. Train a stronger player in 5.4 (on musclemimic_models **1.0.6**), then re-export
6. Optional: **play** `myoChallengeChaseTagFBVs-v0` locally (score + render)

Site: [mc26.myosuite.org](https://mc26.myosuite.org)

Invitation only. If the site says your email is not on the list, [apply here](https://docs.google.com/forms/d/e/1FAIpQLSf0tsCktz3adRUB6YGd0OfQMP9lQDrzeHeZCozrlVDN2J6_qw/viewform) first.


## What you submit

Each team submits **one file: a single player**.

You do **not** submit an opponent. The server always provides the other player.

| You send | The server does |
|---|---|
| **One** ONNX model | Runs it as player A |
| | Puts a champion (or another team) as player B |
| | Scores the match |

The file is a neural net that, every step:

- **reads** 546 numbers — your body, the opponent’s **hip** (not their muscles), and a **chase/evade flag**
- **writes** 354 numbers — your muscle activations, each between 0 and 1

```
obs (546)  →  YOUR MODEL  →  action (354)
```

You do **not** pick chase vs evade. One file plays both. The server assigns the role each episode and writes it into the observation. Opponent hip is only **where they are** — not whether to hunt or flee.

| Slice | What it is |
|---|---|
| `obs[0:537]` | Your body (joints, velocities, muscles, pelvis) |
| `obs[537:544]` | Opponent hip relative pose (where they are) |
| `obs[544:546]` | **Role** — `[1, 0]` = **chase** (go toward them), `[0, 1]` = **evade** (run away) |

Do not infer chase/evade from hip position. The same offset means opposite actions depending on those last two numbers. The export cell already reads `obs[544:546]` and flips heading.

That same file is ranked against a fixed champion during the cycle, then used in a 1v1 tournament against other teams.


## 1. Register on the website

Do this in the browser, not in Colab.

1. Open [mc26.myosuite.org](https://mc26.myosuite.org).
2. Click **Register your team**.
3. Fill **team name**, **email**, and **password** (8+ characters). Country and education are optional.
4. Click **Register team**. Already registered? Use **Sign in**.

<p><img alt="Register and Sign in on mc26.myosuite.org" width="480" src="https://raw.githubusercontent.com/MyoHub/myosuite/dev/tutorials/files/5.6/colab-auth.png"></p>


## 2. Copy your CLI token

On the website, open **Submissions** and copy the **CLI submission token**. Paste it with your email into the Colab cell below. Do not share the token.

<p><img alt="CLI submission token on Submissions" width="440" src="https://raw.githubusercontent.com/MyoHub/myosuite/dev/tutorials/files/5.6/colab-token.png"></p>


## 3. Export the 5.4 walker and submit it

The cells below download the tutorial 5.4 1.0.6 walker (`policy_heading_table.pt`), wrap it for ChaseTag, and upload the ONNX.

The walker reads **528** numbers. The server sends **546**. The export maps 546 → 528: heading from opponent hip XY, **flipped by the role flag** `obs[544:546]` (`[1, 0]` chase toward them, `[0, 1]` evade away). Do not export the raw 528-d net.

If you already trained 5.4, upload your own `policy_heading_table.pt` or `policy_bc_best.pt` into this Colab first — that file is used instead of the download.

Paste email + token, then **Runtime → Run all**.


In [ ]:
# @title Install
%pip -q install -U "evalmc>=0.0.4" onnx
print("ok")


In [ ]:
# @title Paste email + CLI token from the website
EMAIL = ""  # @param {type:"string"}
AUTH_TOKEN = ""  # @param {type:"string"}

EMAIL = EMAIL.strip().lower()
AUTH_TOKEN = AUTH_TOKEN.strip()
if not EMAIL or not AUTH_TOKEN:
    raise SystemExit("Paste EMAIL and AUTH_TOKEN from mc26.myosuite.org → Submissions.")
print("email", EMAIL, "token length", len(AUTH_TOKEN))


In [ ]:
# @title Export tutorial 5.4 .pt → ChaseTag ONNX (546 → 354)
from pathlib import Path
import urllib.request

import torch
import torch.nn as nn

CKPT = Path("policy_heading_table.pt")
ONNX_PATH = Path("chasetag_player.onnx")
GIT_REF = "fix/tutorial-5.4-mm-10m-2-models-version"
CKPT_URLS = [
    Path("policy_heading_table.pt"),
    Path("policy_bc_best.pt"),
    Path("tutorials/files/5.6/policy_heading_table.pt"),
    Path("tutorials/files/5.6/policy_bc_best.pt"),
]
CKPT = next((p for p in CKPT_URLS if p.is_file()), Path("policy_heading_table.pt"))
if not CKPT.is_file():
    last_err = None
    for ref in (GIT_REF, "dev"):
        url = (
            f"https://raw.githubusercontent.com/MyoHub/myosuite/{ref}"
            "/tutorials/files/5.6/policy_heading_table.pt"
        )
        print("Downloading tutorial 5.4 1.0.6 walker …", url)
        try:
            urllib.request.urlretrieve(url, CKPT)
            last_err = None
            break
        except Exception as exc:
            last_err = exc
    if last_err is not None and not CKPT.exists():
        raise last_err
    print("saved", CKPT.resolve(), "bytes", CKPT.stat().st_size)
else:
    print("Using local", CKPT.resolve())


class _SiLULN(nn.Module):
    def __init__(self, dim: int) -> None:
        super().__init__()
        self.fc = nn.Linear(dim, dim)
        self.ln = nn.LayerNorm(dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.nn.functional.silu(self.ln(self.fc(x)))


class ActorCritic(nn.Module):
    def __init__(self, obs_dim=528, act_dim=354, hidden=512, layers=6):
        super().__init__()
        trunk = [nn.Linear(obs_dim, hidden), nn.SiLU()]
        for _ in range(layers - 1):
            trunk.append(_SiLULN(hidden))
        self.trunk = nn.Sequential(*trunk)
        self.actor_mean = nn.Linear(hidden, act_dim)
        self.log_std = nn.Parameter(torch.zeros(act_dim))
        self.critic = nn.Linear(hidden, 1)


class HeadingTablePolicy(nn.Module):
    """528-d walker: BC mean + per-heading left-leg bias (musclemimic_models 1.0.6)."""

    _DIRS = torch.tensor(
        [
            [1.0, 0.0],
            [0.70710678, 0.70710678],
            [0.0, 1.0],
            [-0.70710678, 0.70710678],
            [-1.0, 0.0],
            [-0.70710678, -0.70710678],
            [0.0, -1.0],
            [0.70710678, -0.70710678],
        ]
    )

    def __init__(self, ac: ActorCritic, table: torch.Tensor):
        super().__init__()
        self.trunk = ac.trunk
        self.actor_mean = ac.actor_mean
        self.register_buffer("dirs", self._DIRS.clone())
        self.register_buffer("table", table.float())

    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        base = self.actor_mean(self.trunk(obs))
        idx = torch.argmax(obs[:, 520:522] @ self.dirs.T, dim=-1)
        out = base.clone()
        out[:, 314:354] = torch.clamp(out[:, 314:354] + self.table[idx], 0.0, 1.0)
        return out


class FbvsWalker(nn.Module):
    """FBVs obs 546 → 354. Hip XY heading; role [1,0] chase / [0,1] evade."""

    def __init__(self, walker: nn.Module):
        super().__init__()
        self.walker = walker

    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        qpos, qvel = obs[:, 0:89], obs[:, 89:177]
        act, opp_rel, role = obs[:, 177:531], obs[:, 537:544], obs[:, 544:546]
        w, x, y, z = qpos[:, 3], qpos[:, 4], qpos[:, 5], qpos[:, 6]
        yaw = torch.atan2(2 * (w * z + x * y), 1 - 2 * (y * y + z * z))
        c, s = torch.cos(-yaw), torch.sin(-yaw)
        vx, vy = qvel[:, 0], qvel[:, 1]
        delta = opp_rel[:, :2]
        evade = (role[:, 1:2] > 0.5).to(obs.dtype)
        delta = torch.where(evade > 0.5, -delta, delta)
        n = torch.linalg.norm(delta, dim=-1, keepdim=True).clamp_min(1e-6)
        roll = torch.atan2(2 * (w * x + y * z), 1 - 2 * (x * x + y * y))
        pitch = torch.asin((2 * (w * y - z * x)).clamp(-1.0, 1.0))
        wx_w, wy_w = qvel[:, 3], qvel[:, 4]
        obs528 = torch.cat(
            [
                qpos[:, 7:89],
                qvel[:, 6:],
                act,
                torch.stack([c * vx - s * vy, s * vx + c * vy], dim=-1),
                delta / n,
                torch.stack(
                    [roll, pitch, c * wx_w - s * wy_w, s * wx_w + c * wy_w, qvel[:, 5], qvel[:, 2]],
                    dim=-1,
                ),
            ],
            dim=-1,
        )
        return torch.clamp(self.walker(obs528), 0.0, 1.0)


state = torch.load(CKPT, map_location="cpu", weights_only=True)
if isinstance(state, dict) and state.get("kind") == "heading_table":
    ac = ActorCritic(obs_dim=528)
    ac.load_state_dict(state["actor"])
    walker = HeadingTablePolicy(ac, state["table"]).eval()
    how = "5.4 heading-table walker (1.0.6) + role-aware heading"
elif isinstance(state, dict) and "trunk.0.weight" in state:
    obs_dim = int(state["trunk.0.weight"].shape[1])
    ac = ActorCritic(obs_dim=obs_dim)
    ac.load_state_dict(state)
    if obs_dim == 528:
        class _Mean(nn.Module):
            def __init__(self, ac):
                super().__init__()
                self.ac = ac
            def forward(self, obs):
                return self.ac.actor_mean(self.ac.trunk(obs))
        walker = _Mean(ac).eval()
        how = "5.4 ActorCritic + role-aware heading"
    elif obs_dim == 546:
        class FbvsTrainActor(nn.Module):
            def __init__(self, ac):
                super().__init__()
                self.trunk = ac.trunk
                self.actor_mean = ac.actor_mean
            def forward(self, obs):
                return torch.clamp(self.actor_mean(self.trunk(obs)), 0.0, 1.0)
        wrap = FbvsTrainActor(ac).eval()
        torch.onnx.export(
            wrap, torch.zeros(1, 546), str(ONNX_PATH), opset_version=17,
            input_names=["obs"], output_names=["action"],
            dynamic_axes={"obs": {0: "batch"}, "action": {0: "batch"}},
            do_constant_folding=True, dynamo=False,
        )
        print(f"Wrote {ONNX_PATH.resolve()}")
        print("Shape: obs [batch, 546] → action [batch, 354]  (FBVsTrain 546-d as-is)")
        raise SystemExit
    else:
        raise SystemExit(f"Expected obs_dim 528 or 546, got {obs_dim}")
else:
    raise SystemExit(f"Unrecognized checkpoint keys: {list(state)[:8] if isinstance(state, dict) else type(state)}")

wrap = FbvsWalker(walker).eval()
torch.onnx.export(
    wrap,
    torch.zeros(1, 546),
    str(ONNX_PATH),
    opset_version=17,
    input_names=["obs"],
    output_names=["action"],
    dynamic_axes={"obs": {0: "batch"}, "action": {0: "batch"}},
    do_constant_folding=True,
    dynamo=False,
)
print(f"Wrote {ONNX_PATH.resolve()}")
print(f"Shape: obs [batch, 546] → action [batch, 354]  ({how})")


In [ ]:
# @title Submit
import subprocess
import sys

cmd = [
    sys.executable, "-m", "evalmc", "submit-onnx", str(ONNX_PATH),
    "-t", "chasetag", "-p", "1",
    "-e", EMAIL, "-a", AUTH_TOKEN,
]
print("Uploading", ONNX_PATH.name, "…")
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise SystemExit(proc.returncode)
print("Upload accepted. Now open the website Submissions tab (next section).")


## 4. Watch evaluation on the website

Go back to [mc26.myosuite.org](https://mc26.myosuite.org) → **Submissions**. Refresh until the row moves **submitted → evaluating → scored**.

<p><img alt="Submissions table with status badges" width="440" src="https://raw.githubusercontent.com/MyoHub/myosuite/dev/tutorials/files/5.6/colab-watch.png"></p>

| Status | Meaning |
|---|---|
| **submitted** | queued |
| **evaluating** | running |
| **scored** | done — a number appears under Score |
| **failed** | click the row for the error |

This file is the 5.4 directional walker, not a random dummy. Train further in section 5, then re-run **Export** and **Submit**.


## 5. Train a stronger player

The upload above is the **tutorial 5.4 walker** with heading from opponent hip XY, flipped by the role flag (`[1, 0]` chase, `[0, 1]` evade). ChaseTag uses `musclemimic_models` **1.0.6**; the shipped walker is the 5.4 student adapted onto that body.

To beat it, train your own walker in 5.4, then re-run **Export** and **Submit**.

[![Open tutorial 5.4 in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MyoHub/myosuite/blob/dev/tutorials/5.4_MuscleMimic_Directional_Locomotion.ipynb)

Notebook: [5.4 MuscleMimic directional locomotion](https://github.com/MyoHub/myosuite/blob/dev/tutorials/5.4_MuscleMimic_Directional_Locomotion.ipynb)

1. Train the walker (heading command → 354 muscles).
2. Keep that net. Do **not** export it as-is: it reads **528** numbers; the server sends **546**.
3. Upload your `policy_heading_table.pt` (or `policy_bc_best.pt`) into this Colab and re-run the export cell — it already wraps heading from opponent hip XY using the role flag.

Do **not** fine-tune in `myoChallengeChaseTagFBP2-v0` (537-d, not the 546-d submission contract; see [#576](https://github.com/MyoHub/myosuite/issues/576)).

You still submit **one** file. The server always provides the other player.


## 6. Play a local 1v1 and render a match (optional)

Plays your ONNX file as **both** players of `myoChallengeChaseTagFBVs-v0` (546-d, 1.0.6 body) for a few episodes. Prints how each ended (tag, fall, time) with player A's score. Episode 0 is recorded until it ends.

Leave **Run** unchecked when you only want to submit (`Runtime → Run all`).


In [ ]:
# @title Play myoChallengeChaseTagFBVs-v0 locally and record a match
RUN = False  # @param {type:"boolean"}
ONNX_PATH = "chasetag_player.onnx"  # @param {type:"string"}
EPISODES = 4  # @param {type:"integer"}

if not RUN:
    print("Skipped. Check Run to play the 1v1 env locally and record a match.")
else:
    import os, sys, subprocess
    from pathlib import Path

    if sys.platform != "darwin":
        os.environ.setdefault("MUJOCO_GL", "egl")
        os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
    if "google.colab" in sys.modules:
        subprocess.run(["apt-get", "-qq", "install", "-y", "libegl1", "libgles2"], check=False)
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "-U",
            "git+https://github.com/MyoHub/myosuite.git@fix/tutorial-5.4-mm-10m-2-models-version", "onnxruntime", "imageio", "imageio-ffmpeg",
        ])

    import imageio
    import mujoco
    import numpy as np
    import onnxruntime as ort
    from IPython.display import Video, display
    from myosuite import make_env

    onnx_path = Path(ONNX_PATH.strip())
    if not onnx_path.is_file():
        raise SystemExit(f"{onnx_path} not found: run the export cell or upload your .onnx first.")
    sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    name = sess.get_inputs()[0].name

    def act(obs):
        x = np.asarray(obs, np.float32).reshape(1, -1)
        return np.clip(sess.run(None, {name: x})[0].reshape(-1), 0.0, 1.0)

    env = make_env("myoChallengeChaseTagFBVs-v0")
    raw = env.unwrapped
    # Confirm ChaseTag rebuilt musclemimic_models 1.0.6 (left-knee +0.3695).
    eq = mujoco.mj_name2id(raw.model, mujoco.mjtObj.mjOBJ_EQUALITY, "a0_knee_angle_rotation3_constraint_l")
    knee = float(raw.model.eq_data[eq, 1]) if eq >= 0 else float("nan")
    print(f"env myoChallengeChaseTagFBVs-v0  left-knee coef1={knee:+.4f}  (1.0.6 is +0.3695)")

    renderer = mujoco.Renderer(raw.model, height=360, width=480)
    cam = mujoco.MjvCamera()
    cam.type = mujoco.mjtCamera.mjCAMERA_FREE
    cam.lookat[:] = [0.0, 0.0, 1.0]
    cam.distance = 5.0
    cam.azimuth = 135.0
    cam.elevation = -20.0

    frames = []
    print(f"{'ep':>3}  {'role':>5}  {'steps':>5}  {'t':>6}  {'end':<22}  {'score A':>7}")
    for episode in range(int(EPISODES)):
        obs, info = env.reset(seed=episode)
        n = 0
        while True:
            obs, _, terminated, truncated, info = env.step(
                {"agent_0": act(obs["agent_0"]), "agent_1": act(obs["agent_1"])}
            )
            n += 1
            if episode == 0:
                renderer.update_scene(raw.data, camera=cam)
                frames.append(np.asarray(renderer.render(), dtype=np.uint8))
            if terminated["agent_0"] or truncated["agent_0"]:
                break
        fell = [agent for agent, down in info["fell"].items() if down]
        end = "tag" if info["tagged"] else f"fall ({', '.join(fell)})" if fell else "time limit"
        print(f"{episode:3d}  {info['task']:5s}  {n:5d}  {info['elapsed_s']:5.2f}s  {end:<22}  {info['score']:7.3f}")
    renderer.close()
    env.close()

    out = Path("chasetag_fbvs.mp4")
    with imageio.get_writer(str(out), fps=30, quality=8, macro_block_size=1) as writer:
        for frame in frames[::3]:
            writer.append_data(frame)
    print("episode 0 recorded:", len(frames), "steps ->", out)
    display(Video(str(out), embed=True, html_attributes="controls loop"))
